In [168]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold

#PREPROCESSING:

####Read in raw training set:

In [169]:
DF = pd.read_csv('../data/raw/train.csv')
print(DF.head())

       id             model_a              model_b  \
0   30192  gpt-4-1106-preview           gpt-4-0613   
1   53567           koala-13b           gpt-4-0613   
2   65089  gpt-3.5-turbo-0613       mistral-medium   
3   96401    llama-2-13b-chat  mistral-7b-instruct   
4  198779           koala-13b   gpt-3.5-turbo-0314   

                                              prompt  \
0  ["Is it morally right to try to have a certain...   
1  ["What is the difference between marriage lice...   
2  ["explain function calling. how would you call...   
3  ["How can I create a test set for a very rare ...   
4  ["What is the best way to travel from Tel-Aviv...   

                                          response_a  \
0  ["The question of whether it is morally right ...   
1  ["A marriage license is a legal document that ...   
2  ["Function calling is the process of invoking ...   
3  ["Creating a test set for a very rare category...   
4  ["The best way to travel from Tel Aviv to Jeru...   

 

####Clean & reformat dataset to just include required columns

In [170]:
longDF = pd.concat( [
    DF[['model_a', 'prompt', 'response_a']].rename(columns={'model_a': 'LLM_name', 'prompt': 'LLM_Input', 'response_a': 'LLM_output'}),
    DF[['model_b', 'prompt', 'response_b']].rename(columns={'model_b': 'LLM_name','prompt': 'LLM_Input', 'response_b': 'LLM_output'}),
], ignore_index=True)

print(longDF.head())

DF = longDF

             LLM_name                                          LLM_Input  \
0  gpt-4-1106-preview  ["Is it morally right to try to have a certain...   
1           koala-13b  ["What is the difference between marriage lice...   
2  gpt-3.5-turbo-0613  ["explain function calling. how would you call...   
3    llama-2-13b-chat  ["How can I create a test set for a very rare ...   
4           koala-13b  ["What is the best way to travel from Tel-Aviv...   

                                          LLM_output  
0  ["The question of whether it is morally right ...  
1  ["A marriage license is a legal document that ...  
2  ["Function calling is the process of invoking ...  
3  ["Creating a test set for a very rare category...  
4  ["The best way to travel from Tel Aviv to Jeru...  


####Get counts of each different model

In [171]:
print(DF["LLM_name"].value_counts().to_string())

LLM_name
gpt-4-1106-preview                7387
gpt-3.5-turbo-0613                7083
gpt-4-0613                        6165
claude-2.1                        5583
claude-instant-1                  4136
gpt-4-0314                        4122
claude-1                          3978
vicuna-33b                        3720
mixtral-8x7b-instruct-v0.1        3545
vicuna-13b                        3448
llama-2-70b-chat                  3428
gpt-3.5-turbo-1106                3352
mistral-medium                    3315
llama-2-13b-chat                  2607
claude-2.0                        2456
zephyr-7b-beta                    2401
palm-2                            1977
llama-2-7b-chat                   1793
wizardlm-70b                      1644
openchat-3.5                      1632
mistral-7b-instruct               1617
koala-13b                         1598
vicuna-7b                         1591
wizardlm-13b                      1580
oasst-pythia-12b                  1494
gemini-pro-dev-a

####Narrowed down to just 3 common LLMs: gpt-4-0613, claude-2.1, llama-2-70b-chat

In [172]:
DF = DF.loc[DF['LLM_name'].isin(['gpt-4-0613', 'claude-2.1', 'llama-2-70b-chat'])]
DF.head()

,LLM_name,LLM_Input,LLM_output
20,claude-2.1,"[""Write a python code that calculates sum of 5...","[""Here is the python code that calculates the ..."
38,gpt-4-0613,"[""Hi who are you""]","[""Hello! I'm an AI developed to assist with ta..."
45,gpt-4-0613,"[""write the lyrics to a rap song about some du...","[""(Verse 1)\nYo, let's take a minute, prepare ..."
47,claude-2.1,"[""#include <stdio.h>\n#include <stdlib.h>\n#in...","[""This C program analyzes the contents of a te..."
48,gpt-4-0613,"[""Write OCL statement each Employee working in...","[""context Employee inv:\nself.age > 22""]"


####Create testing/training split using StratifiedKFold to guarantee class proportions with 80/20 split

In [173]:
X = DF[['LLM_Input', 'LLM_output']]
Y = DF['LLM_name']

dataSplit = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(dataSplit.split(X, Y), start=1):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    Y_train = Y.iloc[train_idx]
    Y_test = Y.iloc[test_idx]

    print(f"Fold {fold}: {len(X_train)} train, {len(X_test)} test")
    

X_train.to_csv('../data/processed/XTrain.csv', index=False)
X_test.to_csv('../data/processed/XTest.csv', index=False)
Y_train.to_csv('../data/processed/YTrain.csv', index=False)
Y_test.to_csv('../data/processed/YTest.csv', index=False)

Fold 1: 12140 train, 3036 test
Fold 2: 12141 train, 3035 test
Fold 3: 12141 train, 3035 test
Fold 4: 12141 train, 3035 test
Fold 5: 12141 train, 3035 test


##CNN MODEL

In [174]:
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [175]:
import sys
import platform
import tensorflow as tf

print("Operating system:", platform.system())
print("Python executable:", sys.executable)
print("TensorFlow version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs:", tf.config.list_physical_devices("GPU"))

Operating system: Windows
Python executable: e:\python\python.exe
TensorFlow version: 2.21.0
Built with CUDA: False
GPUs: []


#### Read in data and OHE for target column

In [176]:
tf.keras.utils.set_random_seed(42)

X_train = pd.read_csv('../data/processed/XTrain.csv')
X_test = pd.read_csv('../data/processed/XTest.csv')
Y_train = pd.read_csv('../data/processed/YTrain.csv')
Y_test = pd.read_csv('../data/processed/YTest.csv')

X_train_Input = X_train['LLM_Input'].fillna("").astype(str).to_numpy()
X_train_Output = X_train['LLM_output'].fillna("").astype(str).to_numpy()

X_test_Input = X_test['LLM_Input'].fillna("").astype(str).to_numpy()
X_test_Output = X_test['LLM_output'].fillna("").astype(str).to_numpy()

X_train_Full = ("inputfield " + X_train["LLM_Input"].fillna("").astype(str) + " outputfield " + X_train["LLM_output"].fillna("").astype(str)).to_numpy()
X_test_Full = ("inputfield " + X_test["LLM_Input"].fillna("").astype(str) + " outputfield " + X_test["LLM_output"].fillna("").astype(str)).to_numpy()

encoder = LabelEncoder()
Y_train = encoder.fit_transform(Y_train)
Y_test = encoder.fit_transform(Y_test)




e:\python\Lib\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
e:\python\Lib\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


####Tokenizing LLM responses

In [177]:
MAX_TOKENS = 20000
SEQUENCE_LENGTH = 1000

def createTokenizer(trainingSet):
    vectorizer = tf.keras.layers.TextVectorization(
        max_tokens=MAX_TOKENS,
        standardize="lower_and_strip_punctuation",
        split="whitespace",
        output_mode="int",
        output_sequence_length=SEQUENCE_LENGTH,
    )

    # Fit the vocabulary ONLY on the training subset.
    vectorizer.adapt(
        tf.data.Dataset.from_tensor_slices(trainingSet).batch(64)
    )
    # lengths = pd.Series(trainingSet).str.split().str.len()
    # print(lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
    return vectorizer



In [178]:
def trainModelCNN(vectorizer, kernelSize, lr, filters):
    inputs = tf.keras.Input(shape=(1,), dtype=tf.string, name="text")

    x = vectorizer(inputs)

    x = tf.keras.layers.Embedding(
        input_dim=len(vectorizer.get_vocabulary()),
        output_dim=128,
    )(x)

    x = tf.keras.layers.Conv1D(
        filters=filters,
        kernel_size=kernelSize,
        activation="relu",
        padding="valid",
    )(x)

    x = tf.keras.layers.GlobalMaxPooling1D()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)

    outputs = tf.keras.layers.Dense(
        3,
        activation="softmax",
    )(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    # model.summary()
    return(model)

In [179]:
def trainModelRNN(vectorizer, enabledDense, lr, output_dim):
    inputs = tf.keras.Input(shape=(1,), dtype=tf.string, name="text")

    x = vectorizer(inputs)

    x = tf.keras.layers.Embedding(
        input_dim=len(vectorizer.get_vocabulary()),
        output_dim=output_dim,
        mask_zero=True,
    )(x)

    x = tf.keras.layers.LSTM(
        units=64,
        return_sequences=False
    )(x)

    x = tf.keras.layers.Dropout(0.5)(x)
    if (enabledDense):
        x = tf.keras.layers.Dense(64, activation="relu")(x)

    outputs = tf.keras.layers.Dense(
        3,
        activation="softmax",
    )(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    # model.summary()
    return(model)

In [180]:
def make_dataset(texts, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (texts.reshape(-1, 1), labels)
    )
    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(texts),
            seed=42,
            reshuffle_each_iteration=True,
        )
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)

def applyModel(trainingSet, trainingOutput, validationSet, validationOutput, testingSet, testingOutput, model):

    train_ds = make_dataset(trainingSet, trainingOutput, shuffle=True)
    val_ds = make_dataset(validationSet, validationOutput)
    test_ds = make_dataset(testingSet, testingOutput)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=20,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=3,
                restore_best_weights=True,
            )
        ],
    )
    test_loss, test_accuracy = model.evaluate(test_ds)

    probabilities = model.predict(test_ds)
    predicted_ids = probabilities.argmax(axis=1)

    print("Test accuracy:", test_accuracy)

    print(classification_report(
        Y_test,
        predicted_ids,
        labels=np.arange(3),
        target_names=[str(label) for label in encoder.classes_],
        zero_division=0,
    ))

    print(confusion_matrix(
        Y_test,
        predicted_ids,
        labels=np.arange(3),
    ))

In [181]:
def applyCNN(X_Train, X_Test, Y_Train, Y_Test):
    X_Train, Val_Train, Y_Train, Y_Val = train_test_split(
    X_Train,
    Y_Train,
    test_size=0.2,
    random_state=42, 
    stratify=Y_train)
    
    kernelSizes = [3, 5, 7]
    filters = [64, 128, 256]
    differentLrs = [3e-4, 1e-3, 3e-3]
    
    for i in kernelSizes:
        for j in filters:
            for k in differentLrs:
                print('--------------------------------------')
                print('CNN with:')
                print('Kernel size: ' + str(i))
                print('Filters: ' + str(j))
                print('Learning rate: ' + str(k))
                print('--------------------------------------')
                vectorizer = createTokenizer(X_Train)
                model = trainModelCNN(vectorizer, i, k, j)
                applyModel(X_Train, Y_Train, Val_Train, Y_Val, X_Test, Y_Test, model)
    
def applyRNN(X_Train, X_Test, Y_Train, Y_Test):
    X_Train, Val_Train, Y_Train, Y_Val = train_test_split(
    X_Train,
    Y_Train,
    test_size=0.2,
    random_state=42, 
    stratify=Y_train)
    
    vectorizer = createTokenizer(X_Train)
    
    enabledDense = [False, True]
    outputDims = [64, 128]
    differentLrs = [3e-4, 3e-3]
    
    for i in enabledDense:
        for j in outputDims:
            for k in differentLrs:
                print('--------------------------------------')
                print('RNN with:')
                print('Hidden dense layer: ' + str(i))
                print('Output dimensions: ' + str(j))
                print('Learning rate: ' + str(k))
                print('--------------------------------------')
                model = trainModelRNN(vectorizer, i, k, j)
                applyModel(X_Train, Y_Train, Val_Train, Y_Val, X_Test, Y_Test, model)

In [182]:
applyCNN(X_train_Output, X_test_Output, Y_train, Y_test)


--------------------------------------
CNN with:
Kernel size: 3
Filters: 64
Learning rate: 0.0003
--------------------------------------
Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


290/304 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.5045 - loss: 1.0209

KeyboardInterrupt: 

In [ ]:
applyRNN(X_train_Output, X_test_Output, Y_train, Y_test)

--------------------------------------
RNN with:
Hidden dense layer: False
Output dimensions: 64
Learning rate: 0.0003
--------------------------------------
Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 145s 470ms/step - accuracy: 0.5292 - loss: 0.9906 - val_accuracy: 0.6258 - val_loss: 0.8193
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 149s 491ms/step - accuracy: 0.6941 - loss: 0.6778 - val_accuracy: 0.6986 - val_loss: 0.6707
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 162s 533ms/step - accuracy: 0.8021 - loss: 0.5145 - val_accuracy: 0.7085 - val_loss: 0.6951
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 156s 512ms/step - accuracy: 0.8836 - loss: 0.3367 - val_accuracy: 0.7937 - val_loss: 0.5434
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 156s 513ms/step - accuracy: 0.8274 - loss: 0.4524 - val_accuracy: 0.7756 - val_loss: 0.6518
Epoch 6/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 157s 516ms/step - accuracy: 0.9430 - loss: 0.1826 - val_accuracy: 0.7867 - val_loss: 0.6127
Epoch 7/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 156s 513ms/step - accuracy: 0.9535 - loss: 0.1548 - val_accuracy: 0.7291 - val_loss: 0.6691
95/95 ━━━━━━━━━━━━━━━━━━━━ 14s 148ms/step - accuracy: 0.7769 - loss: 0.5738
95/95 ━━━━━

e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 153s 500ms/step - accuracy: 0.5797 - loss: 0.8971 - val_accuracy: 0.6599 - val_loss: 0.7919
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 152s 500ms/step - accuracy: 0.7447 - loss: 0.6447 - val_accuracy: 0.7040 - val_loss: 0.7531
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 151s 496ms/step - accuracy: 0.8721 - loss: 0.3596 - val_accuracy: 0.6867 - val_loss: 0.8371
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 151s 497ms/step - accuracy: 0.9390 - loss: 0.1881 - val_accuracy: 0.6904 - val_loss: 0.9647
Epoch 5/20
201/304 ━━━━━━━━━━━━━━━━━━━━ 46s 455ms/step - accuracy: 0.9125 - loss: 0.2623

KeyboardInterrupt: 

In [ ]:
applyCNN(X_train_Input, X_test_Input, Y_train, Y_test)

count    9712.000000
mean       62.254325
std       168.945201
min         1.000000
50%        17.000000
90%       139.000000
95%       236.000000
99%       834.340000
max      4036.000000
dtype: float64


Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text (InputLayer)               │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_12           │ (None, 1000)           │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_9 (Embedding)         │ (None, 1000, 128)      │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 996, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_7          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,650,499 (10.11 MB)

 Trainable params: 2,650,499 (10.11 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 19s 59ms/step - accuracy: 0.3974 - loss: 1.0736 - val_accuracy: 0.3985 - val_loss: 1.0732
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 18s 59ms/step - accuracy: 0.4517 - loss: 1.0452 - val_accuracy: 0.3816 - val_loss: 1.0800
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 18s 59ms/step - accuracy: 0.5706 - loss: 0.9302 - val_accuracy: 0.3545 - val_loss: 1.1559
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 18s 59ms/step - accuracy: 0.7226 - loss: 0.7037 - val_accuracy: 0.3532 - val_loss: 1.3147
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.4033 - loss: 1.0731
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
Test accuracy: 0.4032948911190033
                  precision    recall  f1-score   support

      claude-2.1       0.38      0.14      0.20      1116
      gpt-4-0613       0.41      0.87      0.55      1233
llama-2-70b-chat       0.00      0.00      0.00       686

        accuracy                           0.40      3035
       macro avg       0.26      0.34      0.25    

In [ ]:
applyRNN(X_train_Input, X_test_Input, Y_train, Y_test)

count    9712.000000
mean       62.254325
std       168.945201
min         1.000000
50%        17.000000
90%       139.000000
95%       236.000000
99%       834.340000
max      4036.000000
dtype: float64


Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 1000)      │          0 │ text[0][0]        │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_10        │ (None, 1000, 128) │  2,560,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 1000)      │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 64)        │     49,408 │ embedding_10[0][… │
│                     │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 64)        │          0 │ lstm_2[0][0]      │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 64)        │      4,160 │ dropout_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 3)         │        195 │ dense_20[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,613,763 (9.97 MB)

 Trainable params: 2,613,763 (9.97 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 68s 217ms/step - accuracy: 0.4011 - loss: 1.0729 - val_accuracy: 0.4068 - val_loss: 1.0732
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 66s 216ms/step - accuracy: 0.4547 - loss: 1.0265 - val_accuracy: 0.3751 - val_loss: 1.1079
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 66s 216ms/step - accuracy: 0.6357 - loss: 0.8062 - val_accuracy: 0.3466 - val_loss: 1.3304
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 66s 218ms/step - accuracy: 0.7542 - loss: 0.5865 - val_accuracy: 0.3446 - val_loss: 1.5275
95/95 ━━━━━━━━━━━━━━━━━━━━ 6s 68ms/step - accuracy: 0.4066 - loss: 1.0729
95/95 ━━━━━━━━━━━━━━━━━━━━ 7s 69ms/step
Test accuracy: 0.4065897762775421
                  precision    recall  f1-score   support

      claude-2.1       1.00      0.00      0.00      1116
      gpt-4-0613       0.41      1.00      0.58      1233
llama-2-70b-chat       0.00      0.00      0.00       686

        accuracy                           0.41      3035
       macro avg       0.47      0.33      0.19

In [ ]:
applyCNN(X_train_Full, X_test_Full, Y_train, Y_test)

count    9712.000000
mean      267.133031
std       290.913509
min         4.000000
50%       209.000000
90%       506.900000
95%       723.000000
99%      1582.130000
max      4440.000000
dtype: float64


Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text (InputLayer)               │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_14           │ (None, 1000)           │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_11 (Embedding)        │ (None, 1000, 128)      │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 996, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_8          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,650,499 (10.11 MB)

 Trainable params: 2,650,499 (10.11 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 19s 60ms/step - accuracy: 0.6352 - loss: 0.7836 - val_accuracy: 0.7773 - val_loss: 0.5414
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 19s 63ms/step - accuracy: 0.8046 - loss: 0.4671 - val_accuracy: 0.8172 - val_loss: 0.4505
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 18s 60ms/step - accuracy: 0.9151 - loss: 0.2364 - val_accuracy: 0.8312 - val_loss: 0.4528
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 18s 60ms/step - accuracy: 0.9639 - loss: 0.1071 - val_accuracy: 0.8217 - val_loss: 0.5547
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 18s 60ms/step - accuracy: 0.9795 - loss: 0.0632 - val_accuracy: 0.8234 - val_loss: 0.6122
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8066 - loss: 0.4776
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step
Test accuracy: 0.8065897822380066
                  precision    recall  f1-score   support

      claude-2.1       0.95      0.83      0.88      1116
      gpt-4-0613       0.72      0.94      0.81      1233
llama-2-70b-chat       0.84      0.53 

In [ ]:
applyRNN(X_train_Full, X_test_Full, Y_train, Y_test)

count    9712.000000
mean      267.133031
std       290.913509
min         4.000000
50%       209.000000
90%       506.900000
95%       723.000000
99%      1582.130000
max      4440.000000
dtype: float64


Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 1000)      │          0 │ text[0][0]        │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_12        │ (None, 1000, 128) │  2,560,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 1000)      │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 64)        │     49,408 │ embedding_12[0][… │
│                     │                   │            │ not_equal_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 64)        │          0 │ lstm_3[0][0]      │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 64)        │      4,160 │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_25 (Dense)    │ (None, 3)         │        195 │ dense_24[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,613,763 (9.97 MB)

 Trainable params: 2,613,763 (9.97 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 69s 222ms/step - accuracy: 0.5791 - loss: 0.8857 - val_accuracy: 0.6632 - val_loss: 0.7298
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 65s 215ms/step - accuracy: 0.7967 - loss: 0.5208 - val_accuracy: 0.7340 - val_loss: 0.6635
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 65s 214ms/step - accuracy: 0.9035 - loss: 0.2738 - val_accuracy: 0.7246 - val_loss: 0.7525
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 65s 214ms/step - accuracy: 0.9422 - loss: 0.1697 - val_accuracy: 0.7192 - val_loss: 0.9912
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 65s 213ms/step - accuracy: 0.9647 - loss: 0.1132 - val_accuracy: 0.7081 - val_loss: 1.0688
95/95 ━━━━━━━━━━━━━━━━━━━━ 6s 67ms/step - accuracy: 0.7308 - loss: 0.6651
95/95 ━━━━━━━━━━━━━━━━━━━━ 7s 68ms/step
Test accuracy: 0.7308072447776794
                  precision    recall  f1-score   support

      claude-2.1       0.78      0.84      0.81      1116
      gpt-4-0613       0.71      0.76      0.73      1233
llama-2-70b-chat       0.68      

####Testing with different hyper params for tokenization

In [ ]:
MAX_TOKENS = 20000
SEQUENCE_LENGTH = 5000

In [ ]:
applyCNN(X_train_Full, X_test_Full, Y_train, Y_test)

count    9712.000000
mean      267.133031
std       290.913509
min         4.000000
50%       209.000000
90%       506.900000
95%       723.000000
99%      1582.130000
max      4440.000000
dtype: float64


Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text (InputLayer)               │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_16           │ (None, 5000)           │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_13 (Embedding)        │ (None, 5000, 128)      │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (None, 4996, 128)      │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_9          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,650,499 (10.11 MB)

 Trainable params: 2,650,499 (10.11 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 99s 323ms/step - accuracy: 0.6317 - loss: 0.7916 - val_accuracy: 0.7855 - val_loss: 0.5285
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 99s 325ms/step - accuracy: 0.8104 - loss: 0.4559 - val_accuracy: 0.8217 - val_loss: 0.4300
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 99s 325ms/step - accuracy: 0.9091 - loss: 0.2409 - val_accuracy: 0.8230 - val_loss: 0.4749
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 99s 325ms/step - accuracy: 0.9650 - loss: 0.1082 - val_accuracy: 0.8234 - val_loss: 0.5158
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 100s 327ms/step - accuracy: 0.9792 - loss: 0.0665 - val_accuracy: 0.8123 - val_loss: 0.6188
95/95 ━━━━━━━━━━━━━━━━━━━━ 6s 63ms/step - accuracy: 0.8145 - loss: 0.4572
95/95 ━━━━━━━━━━━━━━━━━━━━ 6s 61ms/step
Test accuracy: 0.8144975304603577
                  precision    recall  f1-score   support

      claude-2.1       0.92      0.86      0.89      1116
      gpt-4-0613       0.74      0.91      0.82      1233
llama-2-70b-chat       0.81     

In [ ]:
applyRNN(X_train_Full, X_test_Full, Y_train, Y_test)

count    9712.000000
mean      267.133031
std       290.913509
min         4.000000
50%       209.000000
90%       506.900000
95%       723.000000
99%      1582.130000
max      4440.000000
dtype: float64


Model: "functional_14"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 5000)      │          0 │ text[0][0]        │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_14        │ (None, 5000, 128) │  2,560,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, 5000)      │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ (None, 64)        │     49,408 │ embedding_14[0][… │
│                     │                   │            │ not_equal_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 64)        │          0 │ lstm_4[0][0]      │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_28 (Dense)    │ (None, 64)        │      4,160 │ dropout_14[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_29 (Dense)    │ (None, 3)         │        195 │ dense_28[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,613,763 (9.97 MB)

 Trainable params: 2,613,763 (9.97 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20


e:\python\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


304/304 ━━━━━━━━━━━━━━━━━━━━ 529s 2s/step - accuracy: 0.6047 - loss: 0.8523 - val_accuracy: 0.6921 - val_loss: 0.7138
Epoch 2/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 524s 2s/step - accuracy: 0.8065 - loss: 0.4918 - val_accuracy: 0.7287 - val_loss: 0.6907
Epoch 3/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 524s 2s/step - accuracy: 0.9008 - loss: 0.2751 - val_accuracy: 0.7382 - val_loss: 0.7480
Epoch 4/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 524s 2s/step - accuracy: 0.9507 - loss: 0.1440 - val_accuracy: 0.7262 - val_loss: 1.0555
Epoch 5/20
304/304 ━━━━━━━━━━━━━━━━━━━━ 523s 2s/step - accuracy: 0.9627 - loss: 0.1147 - val_accuracy: 0.6966 - val_loss: 1.0666
95/95 ━━━━━━━━━━━━━━━━━━━━ 35s 369ms/step - accuracy: 0.7269 - loss: 0.6742
95/95 ━━━━━━━━━━━━━━━━━━━━ 35s 366ms/step
Test accuracy: 0.7268533706665039
                  precision    recall  f1-score   support

      claude-2.1       0.82      0.84      0.83      1116
      gpt-4-0613       0.68      0.80      0.74      1233
llama-2-70b-chat       0.65      0.41  